In [3]:
import pandas as pd

df = pd.read_csv("../data/processed/cleaned_news.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (44689, 5)


,title,text,subject,date,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",US_News,"February 13, 2017",0
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,politicsNews,"April 5, 2017",1
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,politicsNews,"September 27, 2017",1
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",News,"May 22, 2017",0
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",politicsNews,"June 24, 2016",1


In [4]:
print(df["label"].value_counts())

label
0    23478
1    21211
Name: count, dtype: int64


In [5]:
df["content"] = df["title"].fillna("") + " " + df["text"].fillna("")

In [6]:
df[["title", "content", "label"]].head()

,title,content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,Donald Trump heads for Scotland to reopen a go...,1


In [7]:
X = df["content"]
y = df["label"]

print("X:", X.shape)
print("y:", y.shape)

X: (44689,)
y: (44689,)


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training articles:", len(X_train))
print("Testing articles:", len(X_test))

Training articles: 35751
Testing articles: 8938


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_df=0.7
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

In [ ]:
print("Number of features:", len(vectorizer.get_feature_names_out()))

Number of features: 111586


In [ ]:
print(vectorizer.get_feature_names_out()[:50])

['00' '000' '0000' '00004' '000048' '000063' '00007' '000270' '00042'
 '0005' '0009' '000938' '000a' '000after' '000although' '000american'
 '000california' '000cases' '000cylvia' '000dillon000' '000ecuador'
 '000florida' '000georgia' '000illegal' '000illinois' '000in' '000jose'
 '000kyrgyzstan' '000michigan' '000new' '000oman' '000s' '000saudi'
 '000south' '000th' '000that' '000the' '000uterine' '001' '00106' '00155'
 '0018' '00193' '001romney' '001st' '002' '0020' '00240' '002singapore'
 '003']


In [ ]:
print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (35751, 111586)
Testing TF-IDF shape: (8938, 111586)


In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


In [ ]:
y_pred = model.predict(X_test_tfidf)

print("Predictions generated:", len(y_pred))

Predictions generated: 8938


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Fake", "Real"]
))

Accuracy: 0.9856791228462743

Classification Report:
              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      4696
        Real       0.98      0.99      0.98      4242

    accuracy                           0.99      8938
   macro avg       0.99      0.99      0.99      8938
weighted avg       0.99      0.99      0.99      8938



In [ ]:
results = pd.DataFrame({
    "text": X_test,
    "actual": y_test,
    "predicted": y_pred
})

wrong_predictions = results[results["actual"] != results["predicted"]]

print("Total incorrect predictions:", len(wrong_predictions))
wrong_predictions.head(10)

Total incorrect predictions: 128


,text,actual,predicted
21035,BREAKING: Iran Publicly Humiliates Obama…Unvei...,0,1
19960,EU LEADERS PLEDGE EXTRA €1 Billion In Aid To R...,0,1
15092,"Obama says Trump immigration move 'cruel,' not...",1,0
22840,Graphic: Supreme Court roundup,1,0
18184,BREAKING NEWS: Reince Priebus Makes Early Retu...,0,1
36683,The Exodus: The State Department’s Entire Sen...,0,1
16105,BREAKING: NORTH KOREA Detains American Student...,0,1
38190,TRUMP IS FINALLY Face-to-Face With Putin…Left ...,0,1
11279,"Obama names first African-American, woman to b...",1,0
6375,"MICHELLE, SASHA AND MALIA Will Join Barack On ...",0,1


In [ ]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC()

svm_model.fit(X_train_tfidf, y_train)

print("LinearSVC training completed!")

LinearSVC training completed!


In [ ]:
svm_pred = svm_model.predict(X_test_tfidf)

print("Predictions generated:", len(svm_pred))

Predictions generated: 8938


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

svm_accuracy = accuracy_score(y_test, svm_pred)

print("LinearSVC Accuracy:", svm_accuracy)

print("\nClassification Report:")
print(classification_report(
    y_test,
    svm_pred,
    target_names=["Fake", "Real"]
))

LinearSVC Accuracy: 0.9950771984784068

Classification Report:
              precision    recall  f1-score   support

        Fake       1.00      0.99      1.00      4696
        Real       0.99      1.00      0.99      4242

    accuracy                           1.00      8938
   macro avg       1.00      1.00      1.00      8938
weighted avg       1.00      1.00      1.00      8938



In [ ]:
import joblib

joblib.dump(svm_model, "../models/fake_news_model.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [ ]:
def predict_news(news):
    news_tfidf = vectorizer.transform([news])
    prediction = svm_model.predict(news_tfidf)[0]

    if prediction == 0:
        return "FAKE"
    else:
        return "REAL"

In [ ]:
test_news = """
The government announced a new policy today to improve public transportation
and reduce congestion in major cities. Officials said the program will begin
next month after consultations with state authorities.
"""

print("Prediction:", predict_news(test_news))

Prediction: REAL


In [ ]:
test_news = """
India announced a new national initiative to expand renewable energy capacity,
with officials saying the program will support solar and wind projects across
several states.
"""

print("Prediction:", predict_news(test_news))

Prediction: FAKE


In [ ]:
def predict_news_with_score(news):
    news_tfidf = vectorizer.transform([news])
    
    prediction = svm_model.predict(news_tfidf)[0]
    score = svm_model.decision_function(news_tfidf)[0]
    
    if prediction == 0:
        result = "FAKE"
    else:
        result = "REAL"
    
    return result, score

In [ ]:
test_news = """
The government announced a new policy today to improve public transportation
and reduce congestion in major cities. Officials said the program will begin
next month after consultations with state authorities.
"""

result, score = predict_news_with_score(test_news)

print("Prediction:", result)
print("Decision score:", score)

Prediction: REAL
Decision score: 0.05836338748737396


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(
    svm_model,
    method="sigmoid",
    cv=3
)

calibrated_model.fit(X_train_tfidf, y_train)

print("Calibration completed!")

Calibration completed!


In [ ]:
probabilities = calibrated_model.predict_proba(X_test_tfidf)

print(probabilities[:5])

[[6.04864252e-06 9.99993951e-01]
 [4.57403139e-06 9.99995426e-01]
 [9.48778786e-05 9.99905122e-01]
 [9.99999872e-01 1.28296651e-07]
 [6.93478582e-04 9.99306521e-01]]


In [ ]:
calibrated_pred = calibrated_model.predict(X_test_tfidf)

calibrated_accuracy = accuracy_score(y_test, calibrated_pred)

print("Calibrated LinearSVC Accuracy:", calibrated_accuracy)

Calibrated LinearSVC Accuracy: 0.9946296710673529


In [ ]:
def predict_news(news):
    news_tfidf = vectorizer.transform([news])
    
    prediction = calibrated_model.predict(news_tfidf)[0]
    probabilities = calibrated_model.predict_proba(news_tfidf)[0]
    
    if prediction == 0:
        result = "FAKE"
        confidence = probabilities[0] * 100
    else:
        result = "REAL"
        confidence = probabilities[1] * 100
    
    return result, confidence

In [ ]:
test_news = """
The government announced a new policy today to improve public transportation
and reduce congestion in major cities. Officials said the program will begin
next month after consultations with state authorities.
"""

result, confidence = predict_news(test_news)

print("Prediction:", result)
print(f"Confidence: {confidence:.2f}%")

NameError: name 'predict_news' is not defined

In [ ]:
joblib.dump(calibrated_model, "../models/calibrated_fake_news_model.pkl")

print("Calibrated model saved successfully!")

NameError: name 'joblib' is not defined